<a href="https://colab.research.google.com/github/AHANIK33/Deep-Learning/blob/main/Untitled22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
from google.colab import files
from PIL import Image
import zipfile
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device : ",device)

Device :  cpu


In [4]:
uploaded = files.upload()

Saving archive (7).zip to archive (7).zip


In [22]:
with zipfile.ZipFile("archive (7).zip","r") as zip_ref:
    zip_ref.extractall("/content")

In [23]:
print(os.listdir("/content/"))

['.config', '\\content', '\\content\\dogs-vs-cats-classification', 'archive (7).zip', 'dogs-vs-cats-classification', 'sample_data']


In [24]:
print(os.listdir("/content/dogs-vs-cats-classification"))

['test', 'validation', 'train', 'dataset_info.csv']


In [25]:
print(os.listdir("/content/dogs-vs-cats-classification/train"))

['dogs', 'cats']


In [27]:
train_dir = "/content/dogs-vs-cats-classification/train"
val_dir   = "/content/dogs-vs-cats-classification/validation"
test_dir  = "/content/dogs-vs-cats-classification/test"

print(os.listdir(train_dir))
print(os.listdir(val_dir))
print(os.listdir(test_dir))

['dogs', 'cats']
['dogs', 'cats']
['dogs', 'cats']


In [28]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [29]:
train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    val_dir,
    transform=val_test_transform
)

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=val_test_transform
)

print("Classes:", train_dataset.classes)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Classes: ['cats', 'dogs']
Train samples: 19943
Validation samples: 2492
Test samples: 2495


In [35]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [36]:
images, labels = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Labels shape:", labels.shape)

Batch shape: torch.Size([32, 3, 64, 64])
Labels shape: torch.Size([32])


In [37]:
class MLPModel(nn.Module):

    def __init__(self):

        super(MLPModel, self).__init__()

        self.model = nn.Sequential(


            nn.Flatten(),


            nn.Linear(3 * 64 * 64, 512),

            nn.ReLU(),

            nn.Dropout(0.5),


            nn.Linear(512, 128),

            nn.ReLU(),

            nn.Dropout(0.3),


            nn.Linear(128, 2)
        )

    def forward(self, x):

        return self.model(x)

In [38]:
model = MLPModel().to(device)

print(model)

MLPModel(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=12288, out_features=512, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.3, inplace=False)
    (7): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [39]:
criterion = nn.CrossEntropyLoss()

In [40]:
learning_rate = 0.001

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

In [41]:
epochs = 10

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


for epoch in range(epochs):



    model.train()

    running_loss = 0.0

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)


        optimizer.zero_grad()


        outputs = model(images)


        loss = criterion(outputs, labels)


        loss.backward()


        optimizer.step()

        running_loss += loss.item()


        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()


    train_loss = running_loss / len(train_loader)

    train_accuracy = 100 * correct / total



    model.eval()

    val_running_loss = 0.0

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()


    val_loss = val_running_loss / len(val_loader)

    val_accuracy = 100 * val_correct / val_total




    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)


    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.2f}%"
    )

Epoch [1/10] Train Loss: 0.7024 Train Acc: 56.75% Val Loss: 0.6628 Val Acc: 59.59%
Epoch [2/10] Train Loss: 0.6650 Train Acc: 59.15% Val Loss: 0.6538 Val Acc: 62.16%
Epoch [3/10] Train Loss: 0.6613 Train Acc: 59.38% Val Loss: 0.6581 Val Acc: 58.55%
Epoch [4/10] Train Loss: 0.6590 Train Acc: 60.25% Val Loss: 0.6477 Val Acc: 61.24%
Epoch [5/10] Train Loss: 0.6575 Train Acc: 60.43% Val Loss: 0.6345 Val Acc: 61.84%
Epoch [6/10] Train Loss: 0.6507 Train Acc: 60.80% Val Loss: 0.6419 Val Acc: 62.72%
Epoch [7/10] Train Loss: 0.6483 Train Acc: 61.34% Val Loss: 0.6457 Val Acc: 60.79%
Epoch [8/10] Train Loss: 0.6499 Train Acc: 61.28% Val Loss: 0.6459 Val Acc: 63.60%
Epoch [9/10] Train Loss: 0.6477 Train Acc: 61.70% Val Loss: 0.6352 Val Acc: 62.44%
Epoch [10/10] Train Loss: 0.6482 Train Acc: 61.78% Val Loss: 0.6382 Val Acc: 63.16%


In [42]:
model.eval()

all_predictions = []
all_actuals = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(
            predicted.cpu().numpy()
        )

        all_actuals.extend(
            labels.cpu().numpy()
        )

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [43]:
accuracy = accuracy_score(
    all_actuals,
    all_predictions
)

print(
    f"Test Accuracy: {accuracy * 100:.2f}%"
)

Test Accuracy: 63.17%


In [44]:
print(
    classification_report(
        all_actuals,
        all_predictions,
        target_names=test_dataset.classes
    )
)

              precision    recall  f1-score   support

        cats       0.68      0.50      0.57      1248
        dogs       0.60      0.77      0.68      1247

    accuracy                           0.63      2495
   macro avg       0.64      0.63      0.62      2495
weighted avg       0.64      0.63      0.62      2495



In [46]:
image_path = "/content/images.jpg"

image = Image.open(image_path).convert("RGB")

image = val_test_transform(image)

# Add batch dimension
image = image.unsqueeze(0)

image = image.to(device)

model.eval()

with torch.no_grad():

    output = model(image)

    _, predicted = torch.max(output, 1)

prediction = test_dataset.classes[
    predicted.item()
]

print("Prediction:", prediction)

Prediction: dogs
